In [ ]:
import os
import sys

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

sys.path.append(os.path.abspath("./"))

print(f"Current work directory: {os.getcwd()}")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
# Load original unscaled data
adata_full = sc.read_h5ad('./Data/h5ad/test.h5ad')

In [ ]:
# Drop old 'condition' column if it exists to avoid confusion
if 'condition' in adata_full.obs.columns:
    adata_full.obs.drop(columns=['condition'], inplace=True)

# Recreate the infection groups based on viral load metrics
pct = adata_full.obs['pct_counts_oc43'].astype(float)
tot = adata_full.obs['total_counts_oc43'].astype(float)

# Thresholds
p_hi = 10.0 # High infection threshold (%)
eps = 0.1 # No infection threshold (%)
t_hi = 10 # Upper limit of total_counts for No threshold

# Conditional Mask
no_mask   = ((pct <= eps) | (tot <= t_hi)).fillna(False)
high_mask = ((pct >= p_hi) & (tot > t_hi)).fillna(False)

labels = np.select(
    [no_mask, high_mask],
    ['No infection', 'High infection'],
    default='Low infection'
)

adata_full.obs['infection_group'] = pd.Categorical(
    labels,
    categories=['No infection','Low infection','High infection'],
    ordered=True
)

print(adata_full.obs['infection_group'].value_counts())

In [ ]:
import joblib

# Load the CatBoost model to get the new expected features
model = joblib.load('./Data/model/Compact_cat_after_tunning.pkl')
expected_features = model.feature_names_

# Some genes might be named differently in adata
# In this case ATP5MF is ATP5J2 in adata, so we temporarily map it back for extraction
genes_to_extract = expected_features

# Extract 'No infection' cells
adata_healthy = adata_full[adata_full.obs['infection_group'] == 'No infection'].copy()

df_healthy = adata_healthy[:, genes_to_extract].to_df()

# Rename ATP5J2 back to ATP5MF to match the model
# No rename needed

# Rearrange columns to match model exactly
df_healthy = df_healthy[expected_features]

In [ ]:
# Save to CSV for the ML environment to use
df_healthy.to_csv('./CSV/OC43_healthy_cells_from_test.csv', index=False)
print("Saved df_healthy to CSV successfully!")

In [ ]:
import session_info

session_info.show()